# Demo — Simulación Monte Carlo de audiencia simulada

Este notebook ejecuta el motor `run_simulation` del backend con un ejemplo
realista y visualiza los resultados: distribución de la tasa de aceptación,
principales objeciones e importancia de características.

**Requisitos:** instalar `backend/requirements.txt` y `notebooks/requirements.txt`.

In [ ]:
import sys, os
# Permitir importar el paquete `app` del backend.
sys.path.insert(0, os.path.abspath('../backend'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from app.sim.monte_carlo import run_simulation

print('Entorno listo.')

## 1. Configuración de la simulación

Idea de ejemplo: una app de finanzas personales para freelancers. Definimos
las características valoradas y la sensibilidad al precio del público objetivo.

In [ ]:
config = {
    'n_iterations': 10000,
    'population_size': 1000,
    'adoption_prob_base': 0.22,
    'feature_weights': {
        'usabilidad': 1.2,
        'automatizacion': 1.0,
        'precio_justo': 0.8,
        'soporte': 0.5,
        'integraciones': 0.9,
    },
    'price_sensitivity': 1.3,
    'noise_distribution': {'type': 'normal', 'params': {'loc': 0.0, 'scale': 1.0}},
    'random_seed': 42,
}

result = run_simulation(config)
result['execution_metrics']

## 2. Métricas agregadas

In [ ]:
acc = result['acceptance_rate']
pur = result['purchase_intent_probability']
print(f"Aceptacion de mercado: {acc['mean']:.1%}  (CI95 {acc['ci_95_lower']:.1%} - {acc['ci_95_upper']:.1%})")
print(f"Intencion de compra:   {pur['mean']:.1%}  (CI95 {pur['ci_95_lower']:.1%} - {pur['ci_95_upper']:.1%})")

In [ ]:
# Distribucion Monte Carlo de la tasa de aceptacion
samples = pd.DataFrame(result['raw_samples'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(samples['acceptance_rate'], bins=40, color='#6c5ce7', alpha=0.85)
ax.axvline(acc['mean'], color='black', linestyle='--', label='media')
ax.axvline(acc['ci_95_lower'], color='red', linestyle=':', label='CI 95%')
ax.axvline(acc['ci_95_upper'], color='red', linestyle=':')
ax.set_title('Distribucion de la tasa de aceptacion')
ax.set_xlabel('Tasa de aceptacion'); ax.set_ylabel('Frecuencia')
ax.legend(); plt.show()

## 3. Principales objeciones

In [ ]:
obj = pd.DataFrame(result['top_objections'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(obj['objection'], obj['frequency'], color='#e17055')
ax.invert_yaxis()
ax.set_title('Frecuencia de objeciones'); ax.set_xlabel('Frecuencia')
plt.show()
obj

## 4. Importancia de caracteristicas

In [ ]:
feat = pd.DataFrame(result['feature_importance'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feat['feature'], feat['importance'], color='#00b894')
ax.invert_yaxis()
ax.set_title('Importancia relativa de caracteristicas'); ax.set_xlabel('Importancia')
plt.show()
feat

## 5. Interpretacion

- **Aceptacion / intencion de compra**: usar el intervalo de confianza, no solo
  la media, para comunicar incertidumbre al equipo de producto.
- **Objeciones**: priorizar mitigaciones para la objecion mas frecuente.
- **Importancia de caracteristicas**: enfocar el roadmap en las de mayor
  sensibilidad sobre la adopcion.

En la siguiente iteracion, estos resultados estadisticos se enriqueceran con
perfiles y respuestas textuales generadas por un LLM (Claude).